# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Kaggle Notebook)
Panel phải: **Accelerator = GPU T4 x2**, **Internet = On**, **Add Input** dataset
`lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`. Không cần Kaggle API key.

In [ ]:
import os, sys
!nvidia-smi -L
REPO_URL = "https://github.com/sown101/K4-Track-4-Day10-VisionLanguageModel.git"
REPO = "/kaggle/working/K4-Track-4-Day10-VisionLanguageModel"
if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
print("cwd:", os.getcwd())
!pip -q install -r requirements-colab.txt

In [ ]:
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
!python -m pytest -q

## 1. Dữ liệu: tải và chia split

In [ ]:
# Dataset đã gắn qua Add Input; prepare_data tự tìm thư mục có images/ và labels-YOLO/
!ls /kaggle/input
!python scripts/prepare_data.py --source /kaggle/input

In [ ]:
import json
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl
from lab.labels import yolo_to_pixel

DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
meta = json.loads((DATA / "splits" / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", meta["dataset_version"])   # -> điền vào REPORT.md

# 1) Số ảnh / box từng lớp mỗi split (Câu 6)
print(f"{'split':6s} {'ảnh':>5s} {'box':>5s} " + " ".join(f"{c:>8s}" for c in CLASSES)
      + f" {'ảnh rỗng':>8s} {'phiên':>6s}")
for s, rows in splits.items():
    cnt = Counter(o["label"] for r in rows for o in r["objects"])
    empty = sum(1 for r in rows if not r["objects"])
    sess = len({r["session"] for r in rows})
    print(f"{s:6s} {len(rows):5d} {sum(cnt.values()):5d} " + " ".join(f"{cnt[c]:8d}" for c in CLASSES)
          + f" {empty:8d} {sess:6d}")

# 2) Không phiên quay nào nằm ở hai split (chống rò rỉ)
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = sess[a] & sess[b]
    print(f"giao session {a}∩{b}: {len(inter)}")
    assert not inter, f"rò rỉ session giữa {a} và {b}: {sorted(inter)[:5]}"
print("sessions:", {s: sorted(v) for s, v in sess.items()})

# 3) Kích thước box (diện tích tương đối) theo lớp — để giải thích lớp nào khó (Câu 9)
for c in CLASSES:
    areas = sorted(o["yolo"][2] * o["yolo"][3] for r in splits["train"] for o in r["objects"]
                   if o["label"] == c)
    print(f"{c:8s} diện tích trung vị = {100 * areas[len(areas) // 2]:.2f}% khung hình (n={len(areas)})")

# 4) Vẽ 6 ảnh train có box
COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def draw_gt(rec, ax, title=None):
    img = Image.open(DATA / rec["image_path"]).convert("RGB")
    d = ImageDraw.Draw(img)
    for o in rec["objects"]:
        box = yolo_to_pixel(o["yolo"], rec["width"], rec["height"])
        d.rectangle(box, outline=COLORS[o["label"]], width=2)
        d.text((box[0] + 2, max(box[1] - 11, 0)), o["label"], fill=COLORS[o["label"]])
    ax.imshow(img); ax.axis("off")
    ax.set_title(title or rec["image_id"], fontsize=8)

samples = [r for r in splits["train"] if len({o["label"] for o in r["objects"]}) >= 2][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for rec, ax in zip(samples, axes.flat):
    draw_gt(rec, ax, f'{rec["image_id"]} ({len(rec["objects"])} box)')
plt.tight_layout()
Path("report/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("report/figures/gt_samples.png", dpi=110, bbox_inches="tight")
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.labels import build_prompt, objects_to_answer, objects_to_boxes, box_to_pixel
from scripts.score import iou_xyxy, parse_output

PROMPT = build_prompt("norm1000")
print(PROMPT, "\n")

rec = next(r for r in splits["train"] if len(r["objects"]) >= 3)
ans = objects_to_answer(rec["objects"])
print(rec["image_id"], "->", ans)
assert parse_output(ans) is not None, "score.py không parse được đáp án!"

# Round-trip trên TOÀN BỘ box: YOLO -> 0–1000 (int) -> pixel, so với box gốc
max_err, min_iou, bumped, n = 0.0, 1.0, 0, 0
all_rows = [r for rows in splits.values() for r in rows]
for r in all_rows:
    W, H = r["width"], r["height"]
    assert parse_output(objects_to_answer(r["objects"], "norm1000", W, H)) is not None
    for o in r["objects"]:
        b = objects_to_boxes([o], "norm1000", W, H)[0]["bbox_2d"]
        px = box_to_pixel(b, "norm1000", W, H)
        xc, yc, w, h = o["yolo"]
        orig = ((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H)
        max_err = max(max_err, *(abs(p - q) for p, q in zip(px, orig)))
        min_iou = min(min_iou, iou_xyxy(px, orig))
        bumped += (round((xc + w / 2) * 1000) - round((xc - w / 2) * 1000) < 1
                   or round((yc + h / 2) * 1000) - round((yc - h / 2) * 1000) < 1)
        n += 1
print(f"{n} box | sai số round-trip lớn nhất = {max_err:.2f}px | IoU nhỏ nhất = {min_iou:.3f} "
      f"| box phải nới 1 đơn vị = {bumped}")
# Sai số > 0.5 px chỉ đến từ box bị kẹp vào [0,1000] (vượt mép ảnh) hoặc box bị nới

# Vẽ box đáp án (quy ngược về pixel) chồng lên box gốc: phải trùng nhau
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
draw_gt(rec, axes[0], "GT (YOLO)")
img = Image.open(DATA / rec["image_path"]).convert("RGB"); d = ImageDraw.Draw(img)
for b in json.loads(ans):
    d.rectangle(box_to_pixel(b["bbox_2d"], "norm1000", rec["width"], rec["height"]),
                outline=COLORS[b["label"]], width=2)
axes[1].imshow(img); axes[1].axis("off"); axes[1].set_title("JSON 0–1000 → pixel")
plt.tight_layout(); plt.show()

# Độ dài đáp án (ký tự) — để chọn max_new_tokens ở bước 3
lens = sorted(len(objects_to_answer(r["objects"])) for r in splits["train"])
print("độ dài đáp án: trung vị", lens[len(lens) // 2], "| p99", lens[int(.99 * len(lens))], "| max", lens[-1])

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9